# Week 1 · Environment & Tools: Python, Git, NumPy, and the Seed Habit

# Requirements: pip install numpy
# (Python 3.10+ recommended. Git is a system tool, not a pip package.)

This notebook is your **environment self-check**: it verifies Python, Git, the core
libraries, the `zoro` package, and, most importantly, the deterministic-seed habit
that every later week depends on. It ends by printing a single **readiness score**
(0 to 6), one point per check. Get it to 6 before you move on.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1])) # repo root
# Robust fallback: walk up to the real repo root if the kernel's cwd differs.
p = pathlib.Path.cwd()
while not (p / "zoro").is_dir() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

# Check 1, Python 3.10+.
import platform
print("Python:", sys.version.split()[0], "| platform:", platform.platform())
CHECK_PYTHON = 1 if sys.version_info >= (3, 10) else 0
print("OK: Python 3.10+" if CHECK_PYTHON else "FAIL: need Python 3.10+")

# Check 2, numpy and the zoro package import cleanly.
CHECK_LIBS = 0
try:
    import numpy as np
    import zoro
    from zoro import data
    print("numpy:", np.__version__, "| zoro:", zoro.__version__)
    CHECK_LIBS = 1
except Exception as e:
    print("FAIL: could not import numpy/zoro:", e)

### Why the environment matters first

Andrew Ng's skills map puts **software engineering fundamentals** second on its list,
and Zorost's sharper version of that skill is blunt: when a coding agent writes code,
a hundred tradeoffs are made in ninety seconds and handed back as a diff you must
review. You cannot review what you cannot run. A reproducible environment, pinned
interpreter, pinned dependencies, a clean Git checkout, is the floor under everything
else in this program. The checks above make that floor *measurable*: either the number
is 6, or something is broken.

In [ ]:
# Check 3, zoro.data exposes the seven generators the whole program reuses.
CHECK_ZORO = 0
expected = ["carriers", "lanes", "shipments", "support_tickets", "policy_docs", "bol_samples", "save_all"]
try:
    missing = [f for f in expected if not hasattr(data, f)]
    car = data.carriers(20, seed=7)
    has_cols = set(car.columns) >= {"carrier_id", "carrier_name", "on_time_rate"}
    CHECK_ZORO = 1 if (not missing and has_cols) else 0
    print("zoro.data functions present:", not missing, "| missing:", missing)
    print("carriers(20, seed=7) ->", car.shape[0], "rows x", car.shape[1], "cols")
    print("OK: zoro.data generators callable." if CHECK_ZORO else "FAIL: zoro.data incomplete.")
except Exception as e:
    print("FAIL: zoro.data check raised:", e)

### Check 4: Git is installed and you are inside the repo

Git is the reproducibility layer for *your work* the way seeds are for *your data*.
The three checks below (git present, inside a work tree, on a branch) are the bare
minimum you need before the Week 1 Friday use case, where you commit the generated
dataset.

In [ ]:
import subprocess

def run(args):
    return subprocess.run(args, capture_output=True, text=True)

CHECK_GIT = 0
try:
    gv = run(["git", "--version"])
    inside = run(["git", "rev-parse", "--is-inside-work-tree"]).stdout.strip() == "true"
    branch = run(["git", "branch", "--show-current"]).stdout.strip()
    print("git:", gv.stdout.strip())
    print("inside repo:", inside, "| branch:", branch)
    CHECK_GIT = 1 if (gv.returncode == 0 and inside) else 0
    print("OK: git present and inside a repository." if CHECK_GIT else "FAIL: git missing or not in a repo.")
except Exception as e:
    print("FAIL: git check raised:", e)

### NumPy refresher: the array is the unit of thought

Nearly every model in this program flows through a multidimensional array. Here are
the four moves that cover 90% of real use: create an array, inspect its
`shape`/`dtype`, reduce along an axis, and **broadcast** (align a smaller array against
a larger one along a matching axis).

In [ ]:
rng = np.random.default_rng(0)  # seed 0 for this demo only
a = rng.integers(0, 10, size=(3, 4))
print("array:")
print(a)
print("shape:", a.shape, "| dtype:", a.dtype)
print("column means:", a.mean(axis=0).round(3))

col_offsets = np.array([100, 200, 300, 400])
print("broadcast add (3,4) + (4,):")
print(a + col_offsets)

mask = a > 5
print("elements > 5:", int(mask.sum()), "of", a.size)

CHECK_NUMPY = 1
print("OK: numpy array/broadcast demo ran.")

### The seed habit: the single most important line in this program

AI outputs are unpredictable; that is the fact the whole discipline responds to. The
counterweight is **determinism everywhere you can control it**. A *seed* is a fixed
integer that makes a pseudorandom generator emit the same sequence on every run and
every machine. `numpy.random.default_rng(seed)` is the modern, recommended form. Same
seed, same data, that is what makes a Week 1 dataset reproducible in Week 23.

In [ ]:
def first_five(seed):
    return np.random.default_rng(seed).integers(0, 1_000_000, size=5).tolist()

a = first_five(42)
b = first_five(42)
c = first_five(7)
print("seed 42 run 1:", a)
print("seed 42 run 2:", b)
print("seed  7 run  :", c)
print("same seed, same data:", a == b, "| different seed, different data:", a != c)
CHECK_SEED = 1 if a == b and a != c else 0
print("OK: determinism confirmed." if CHECK_SEED else "FAIL: determinism broken.")

### The metric habit: even a self-check ends in a number

From Week 3 onward every AI artifact ships with a score. Start now: this notebook's
score is the number of checks that passed, out of six.

In [ ]:
READINESS = CHECK_PYTHON + CHECK_LIBS + CHECK_ZORO + CHECK_GIT + CHECK_NUMPY + CHECK_SEED
print("READINESS_SCORE:", READINESS)
if READINESS == 6:
    print("Environment ready. Move to 02-zorologistics-data-generator.ipynb.")
else:
    print("Some checks failed - fix them before continuing.")